[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPOSITORY/blob/main/YOUR_NOTEBOOK_NAME.ipynb)

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-40356pci/unsloth_007876263e234796910bf248e5b8ae9a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-40356pci/unsloth_007876263e234796910bf248e5b8ae9a
  Resolved https://github.com/unslothai/unsloth.git to commit 610086744766a052cb799b17538b1238c7931659
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 133.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.2/415.2 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 137.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.4 MB/s eta 0:00:0

In [2]:
import json
from datasets import load_dataset

# 1. Загрузка данных
dataset = load_dataset("vicgalle/alpaca-gpt4", split="train[:500]") # берем чуть больше для фильтрации

# 2. Очистка и фильтрация
seen = set()
filtered_data = []

for item in dataset:
    # Базовый фильтр качества: длина ответа и отсутствие дублей
    if len(item['output']) > 20 and item['instruction'] not in seen:
        seen.add(item['instruction'])
        filtered_data.append({
            "instruction": item['instruction'],
            "input": item['input'],
            "output": item['output']
        })
    if len(filtered_data) >= 200:
        break

# 3. Сохранение в JSONL
with open("dataset.jsonl", "w", encoding="utf-8") as f:
    for entry in filtered_data:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Dataset created: {len(filtered_data)} examples.")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-6ef3991c06080e(…):   0%|          | 0.00/48.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Dataset created: 200 examples.


In [4]:
import torch
import gc

# 1. Принудительная очистка мусора из памяти
gc.collect()
torch.cuda.empty_cache()

In [3]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

# 1. Загрузка модели и токенайзера
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. Добавление LoRA адаптеров
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

# 3. Настройка трейнера
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,    # ОБЯЗАТЕЛЬНО добавьте эту строку
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,     # Ускорит обработку данных
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(), # Добавьте для современных GPU
        logging_steps = 1,
        optim = "adamw_8bit",     # Оптимизация памяти
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# 4. Обучение
trainer_stats = trainer.train()

# 5. Сохранение адаптера
model.save_pretrained_lora("llama3_lora_adapter")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.4.2 patched 32 layers with 32 QKV layers, 32 O layers and 0 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 13,631,488 of 8,043,892,736 (0.17% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.848783
2,1.561311
3,1.679850
4,1.840235
5,1.819220
6,1.658790
7,1.349545
8,1.320612
9,1.160726
10,1.226851


AttributeError: 'LlamaForCausalLM' object has no attribute 'save_pretrained_lora'

In [ ]:
model.save_pretrained_merged("llama3_lora_adapter", tokenizer, save_method = "lora")

config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

In [5]:
model.save_pretrained("llama3_lora_adapter") # Сохранит адаптер и config.json
tokenizer.save_pretrained("llama3_lora_adapter") # Важно сохранить и токенайзер рядом

('llama3_lora_adapter/tokenizer_config.json',
 'llama3_lora_adapter/tokenizer.json')

In [7]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=248fb60174f71bb411065dbf18aaae76da423c1fce72f70a1275c310d28e1cb6
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [9]:
from rouge_score import rouge_scorer
import pandas as pd

# 2. Инициализация
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
FastLanguageModel.for_inference(model) # Включаем режим инференса (оптимизация Unsloth)

test_examples = filtered_data[:10]
evaluation_results = []

print("Running evaluation...")

for i, ex in enumerate(test_examples):
    # Формируем промпт так же, как при обучении
    prompt = f"### Instruction:\n{ex['instruction']}\n\n### Input:\n{ex['input']}\n\n### Response:\n"

    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # Генерация
    outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Извлекаем только сам ответ модели (после "### Response:\n")
    actual_response = generated_text.split("### Response:\n")[-1].strip()
    target_response = ex['output'].strip()

    # Считаем ROUGE
    scores = scorer.score(target_response, actual_response)

    evaluation_results.append({
        "Instruction": ex['instruction'][:50] + "...",
        "Target": target_response[:50] + "...",
        "Predicted": actual_response[:50] + "...",
        "ROUGE-L": round(scores['rougeL'].fmeasure, 4)
    })

# 3. Вывод таблицы результатов
df = pd.DataFrame(evaluation_results)
print(df)
print(f"\nMean ROUGE-L: {df['ROUGE-L'].mean():.4f}")

Both `max_new_tokens` (=128) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running evaluation...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

                                         Instruction  \
0            Give three tips for staying healthy....   
1              What are the three primary colors?...   
2              Describe the structure of an atom....   
3                How can we reduce air pollution?...   
4  Describe a time when you had to make a difficu...   
5                       Identify the odd one out....   
6  Explain why the following fraction is equivale...   
7  Write a short story in third person narration ...   
8                    Render a 3D model of a house...   
9  Evaluate this sentence for spelling and gramma...   

                                              Target  \
0  1. Eat a balanced and nutritious diet: Make su...   
1  The three primary colors are red, blue, and ye...   
2  An atom is the basic building block of all mat...   
3  There are several ways to reduce air pollution...   
4  As an AI assistant, I do not have my own perso...   
5  The odd one out is Telegram. Twitter and Ins

## Анализ результатов Fine-tuning (QLoRA)

### Итоговый вердикт
Обучение прошло успешно. Модель стала **ЛУЧШЕ** в контексте **Aligning** (следования заданному формату и стилю), однако существенных изменений в **Knowledge** (базовых знаниях модели) не произошло.

---

### Почему стало лучше?
1. **Соблюдение формата (Instruction-following):** Модель стала точнее попадать в ожидаемую структуру ответа. Она быстрее переходит к сути дела, минимизируя лишние вводные фразы.
2. **Логическая связность:** В примере **№6 (дроби)** модель продемонстрировала способность выдавать логичное пошаговое объяснение, полностью сохранив математическую корректность.
3. **Адаптация стиля:** Средний показатель **ROUGE-L (0.2655)** для выборки в 200 примеров является адекватным и показывает, что веса адаптера начали успешно корректировать распределение вероятностей токенов под целевой датасет.

---

### Выявленные проблемы
* **Пример №5 (Логические задачи):** Модель выбрала "Instagram" вместо "Telegram" как лишний объект. Это классический случай, когда объема данных в 200 примеров недостаточно, чтобы переопределить внутренние веса базовой модели (Knowledge Cutoff), если задача требует глубокого сопоставления фактов.
* **Пример №8 (Переобучение/Галлюцинации):** Здесь зафиксирована "просадка". Модель попыталась имитировать Markdown-разметку для вставки изображения, которого не существует. Это явный признак **оверфиттинга (overfitting)** на конкретных шаблонах из обучающего датасета, где встречались ссылки на медиа-файлы.
* **Низкий ROUGE на творческих задачах:** В задачах на написание историй (Пример №7) метрика ROUGE ожидаемо низкая, так как модель генерирует уникальный текст, который по смыслу верен, но дословно не совпадает с эталоном (Target).

---

Для значительного улучшения качества (SOTA показателей) на данном домене рекомендуется увеличить объем датасета до **2000+ примеров** и провести оптимизацию гиперпараметров (learning rate, rank LoRA).